# ANAADHI — LOC-016 Root Correction Pass 2

Forest-Edge Barter Hamlet — canonical environment identity only.

Pass 1 rejected all four candidates. This pass forces a **multi-stall barter hamlet** with repeated weighing infrastructure and excludes animals, lone huts and empty canopies.

Use **Tesla T4**. Run All.


In [ ]:
import sys, subprocess, importlib.util
required = ['diffusers','transformers','accelerate','safetensors','peft']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',*missing])

import torch
from pathlib import Path
from PIL import Image, ImageDraw
from diffusers import AutoPipelineForText2Image
from IPython.display import display

assert torch.cuda.is_available(), 'Enable Kaggle GPU T4 first.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
if 'P100' in GPU_NAME.upper():
    raise RuntimeError('Select Tesla T4 instead of P100.')


In [ ]:
JOB_ID = 'LOC-016'
CANONICAL_NAME = 'Forest-Edge Barter Hamlet'
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'photorealistic Karnataka forest-edge barter hamlet, wide overview of six timber market stalls along muddy laterite clearing, hanging brass balance scales clearly visible, mechanical weighing arms on stall counters, overlapping patchwork solar-cloth awnings, dense Western Ghats forest behind, practical rural materials, restrained future technology, no people'
NEGATIVE_PROMPT = 'people, animals, elephant, isolated hut, lone stall, single shelter, empty canopy, empty field, missing scales, missing weighing arms, missing awnings, resort, cyberpunk, fantasy, concept art, diagram, text, watermark, black bars, blurry'
SEEDS = [1611,1612,1613,1614]
STEPS = 32
GUIDANCE = 6.5
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608
OUTPUT_DIR = Path('/kaggle/working/anaadhi_location_roots/LOC-016-PASS2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(JOB_ID, CANONICAL_NAME, SEEDS)


In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=torch.float16, use_safetensors=True)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_slicing()
except Exception:
    pass
print('Model ready:', MODEL_ID)

def count(tok, text):
    return len(tok(text, truncation=False, add_special_tokens=True).input_ids)
for label,text in [('positive',PROMPT),('negative',NEGATIVE_PROMPT)]:
    c1=count(pipe.tokenizer,text)
    c2=count(pipe.tokenizer_2,text) if getattr(pipe,'tokenizer_2',None) else c1
    print(label,'tokens:',c1,c2)
    if c1>77 or c2>77:
        raise RuntimeError(label+' prompt exceeds 77 tokens')


In [ ]:
target_ratio = MASTER_WIDTH / MASTER_HEIGHT
previews=[]

def crop_scope(img):
    h=round(img.width/target_ratio)
    if h<=img.height:
        y=(img.height-h)//2
        return img.crop((0,y,img.width,y+h))
    w=round(img.height*target_ratio)
    x=(img.width-w)//2
    return img.crop((x,0,x+w,img.height))

for seed in SEEDS:
    print('Generating',seed)
    gen=torch.Generator(device='cpu').manual_seed(seed)
    img=pipe(prompt=PROMPT,negative_prompt=NEGATIVE_PROMPT,width=GEN_WIDTH,height=GEN_HEIGHT,num_inference_steps=STEPS,guidance_scale=GUIDANCE,generator=gen).images[0]
    img=crop_scope(img)
    path=OUTPUT_DIR/f'{JOB_ID}_ST00_PASS2_seed{seed}.png'
    img.save(path)
    previews.append((seed,img.copy()))
    print('Saved:',path,'size:',img.size)
    display(img)

tw=672
th=round(tw/target_ratio)
lh=34
sheet=Image.new('RGB',(tw*2,(th+lh)*2),'white')
draw=ImageDraw.Draw(sheet)
for i,(seed,img) in enumerate(previews):
    t=img.resize((tw,th))
    x=(i%2)*tw
    y=(i//2)*(th+lh)
    sheet.paste(t,(x,y))
    draw.text((x+8,y+th+8),f'LOC-016 PASS2 seed {seed}',fill='black')
sheet_path=OUTPUT_DIR/'LOC-016_PASS2_CONTACT_SHEET.jpg'
sheet.save(sheet_path,quality=92)
print('Finished 4 candidates. Contact sheet:',sheet_path)
display(sheet)


## Review gate
Send the contact sheet to ChatGPT. No candidate is approved automatically.
